In [1]:
import pandas as pd
a = pd.read_csv("hybrid_summary_results_12000.csv")
a.head(10)

,title,context,hybrid_summary
0,"월드 랩스, '세계 생성' 능력 강화한 월드 모델 '마블 1.1' 출시","페이페이 리의 월드 랩스가 새로운 3D 세계 생성 모델을 공개하며, 3D 생성 기술...","페이페이 리의 월드 랩스가 새로운 3D 세계 생성 모델을 공개하며, 3D 생성 기술..."
1,"넷플릭스, 물리 기반 AI 영상 모델 '보이드' 오픈소스 공개","넷플릭스가 영상 속 특정 객체를 제거할 때 단순한 배경 복원이 아니라, 물리적으로 ...","넷플릭스가 영상 속 특정 객체를 제거할 때 단순한 배경 복원이 아니라, 물리적으로 ..."
2,"박정국 엘리스그룹 CTO '데이터센터, 규모보다 운영 역량이 더 중요'","“AI 데이터센터의 성패는 '규모'가 아니라, 얼마나 빨리 짓고 GPU 자원을 10...",하지만 기존 빌딩형 데이터센터는 조 단위 천문학적인 비용과 3년 이상의 건축 기간으...
3,"고재준 KTR 사업단장 “국내 최초 AI 인증, 기업 경쟁력에 날개 달아줄 것”",AI 도입이 가속화되는 가운데 신뢰성 확보가 기업의 핵심 과제로 부상하고 있다. 잘...,KTR(한국화학융합시험연구원)은 AI의 특성을 고려해 국내 최초로 국제 표준 기반 ...
4,이란 공격으로 바레인·두바이의 AWS 데이터센터 마비,이란이 중동 지역의 주요 데이터센터를 타격하면서 글로벌 클라우드 서비스와 반도체 공...,이란이 중동 지역의 주요 데이터센터를 타격하면서 글로벌 클라우드 서비스와 반도체 공...
5,데이터 전문 머코어 해킹으로 주요 AI 기업 '학습 비밀' 유출 비상,세계 최고 수준의 AI 모델 학습 기밀이 담긴 설계도가 유출됐을 가능성이 제기됐다....,이번 사태 이후 유명 해커그룹 랩서스(Lapsus$)는 머코어 유출본이라고 주장하는...
6,클로드 월 구독제로 '오픈클로' 못 쓴다...'시스템에 과도한 부담',앤트로픽이 수요 급증에 대응하기 위해 인기 AI 에이전트 '오픈클로'에 대한 '클로...,앤트로픽이 수요 급증에 대응하기 위해 인기 AI 에이전트 '오픈클로'에 대한 '클로...
7,"아르시, 추론 모델 ‘트리니티-라지-싱킹’ 출시...에이전트 성능은 '클로드급'",미국의 대표 오픈소스 AI 스타트업인 아르시 AI(Arcee AI)가 다단계 추론이...,미국의 대표 오픈소스 AI 스타트업인 아르시 AI(Arcee AI)가 다단계 추론이...
8,"알리바바, 차세대 영상 모델 ‘완2.7-비디오’ 공개","알리바바가 영상 생성부터 편집, 재구성까지 실제 영상 제작 전 과정을 통합한 차세대...","알리바바가 영상 생성부터 편집, 재구성까지 실제 영상 제작 전 과정을 통합한 차세대..."
9,"MS, 일본 AI 시장 확대 위해 15조 투자...AI 인재 100만명 양성",마이크로소프트(MS)가 일본 AI 시장 공략을 위해 대규모 투자에 나선다. MS는 ...,마이크로소프트(MS)가 일본 AI 시장 공략을 위해 대규모 투자에 나선다. MS는 ...


In [2]:
import ollama
import time
import re

test_df = a.head(30).copy()
total_count = len(test_df)
llm_summaries = []

print(f"총 {total_count}개의 요약 작업을 [파이썬 스트리밍 + 앵무새 무한루트 완벽 차단] 모드로 시작합니다!\n" + "="*50)

for idx, row in test_df.iterrows():
    text = str(row['context'])
    title_text = row['title'] if 'title' in row else f"Index {idx}"
    
    if len(text.strip()) == 0:
        continue
        
    prompt = f"""다음 기사를 정확히 3개의 문장으로 요약하라.
[출력 규칙]
- 반드시 3줄만 출력한다.
- 각 줄은 하나의 완전한 문장만 포함한다.
- 줄 구분은 단일 개행 문자(\n)만 사용한다.
- 줄 사이에 빈 줄을 절대 넣지 않는다.
- 번호, 기호, 불릿, 설명을 절대 포함하지 않는다.
- 첫 줄 앞과 마지막 줄 뒤에 공백이나 개행을 넣지 않는다.
- 출력은 오직 요약문 3줄만 작성한다.
{text}"""

    print(f"\n▶ [{idx + 1}/{total_count}] 요약 중: {title_text}")
    print("🤖 상태: ", end="", flush=True)
    
    try:
        response_stream = ollama.chat(
            model='gemma3:27b',
            messages=[{'role': 'user', 'content': prompt}],
            options={
                'temperature': 0.5, 
                'top_p': 0.9,
                'repeat_penalty': 1.1 
            },
            stream=True 
        )
        
        is_thinking = False
        buffer = ""
        summary_chunks = []
        chunk_count = 0
        is_broken = False 
        
        for chunk in response_stream:
            text_chunk = chunk['message']['content']
            buffer += text_chunk
            buffer_lower = buffer.lower()
            
            # 🔥 [강력한 예외 처리] 정규식을 활용한 "모든 형태의 무한루트 버그" 완벽 차단! 🔥
            
            # 조건 1: 2글자 이상의 패턴을 4번 연속 앵무새처럼 읊을 때 (예: "_er_er_er_er", "따라 l서_따라 l서_따라 l서_따라 l서")
            if re.search(r'(.{2,})\1{3,}', buffer_lower):
                print(f"\n🚨 [경고] 모델 뇌정지(패턴 앵무새 무한루트) 감지! 강제로 뚝 끊고 스킵합니다.")
                is_broken = True
                break 
                
            # 조건 2: 1글자를 10번 연속 바보같이 누르고 있을 때 (예: "----------", "가가가가가가가가")
            if re.search(r'(.)\1{9,}', buffer_lower):
                print(f"\n🚨 [경고] 단일 글자 도배(무한루트) 감지! 강제로 뚝 끊고 스킵합니다.")
                is_broken = True
                break

            # 영어 "생각" 가리기 및 본문 스트리밍
            if not is_thinking:
                if "<think>" in buffer_lower or "# 내용 요thought" in buffer_lower:
                    is_thinking = True
                    print("🤔 AI가 깊이 고민(추론) 중입니다", end="", flush=True)
                    buffer = "" 
                elif "<" in buffer and len(buffer) < 15:
                    pass
                elif "#" in buffer and len(buffer) < 15: 
                    pass
                elif "도움이 됩니다" in buffer:
                    buffer = buffer.replace("도움이 됩니다", "")
                else:
                    # 실제 화면 타자 치기 출력
                    print(buffer, end="", flush=True)
                    summary_chunks.append(buffer)
                    buffer = ""
            else:
                chunk_count += 1
                if chunk_count % 100 == 0:  
                    print(".", end="", flush=True)
                    
                if "</think>" in buffer_lower or "<channel|>" in buffer_lower or "final output" in buffer_lower or "done thinking" in buffer_lower:
                    is_thinking = False
                    print("\n✨ 고민 완료! 실제 요약 결과:\n", end="", flush=True) 
                    
                    if "</think>" in buffer_lower:
                        split_idx = buffer_lower.find("</think>") + len("</think>")
                    elif "<channel|>" in buffer_lower:
                        split_idx = buffer_lower.find("<channel|>\n") + len("<channel|>\n")
                    elif "done thinking" in buffer_lower:
                        split_idx = buffer_lower.find("done thinking") + len("done thinking")
                    else:
                        split_idx = buffer_lower.find("final output") + len("final output")
                        
                    real_text = buffer[split_idx:]
                    real_text = real_text.replace("construction:", "").replace(":", "")
                    if real_text.startswith('\n') or real_text.startswith('.'):
                        real_text = real_text.lstrip('\n. ')
                        
                    print(real_text, end="", flush=True)
                    summary_chunks.append(real_text)
                    buffer = ""
        
        # 모델이 고장나서 for문이 중간에 뚝 끊긴 경우
        if is_broken:
            llm_summaries.append("추출 실패(무한루트 발생)")
            print("-" * 50)
            continue # 다음 작업으로 시원하게 Skip
            
        # 아무 문제 없이 깔끔하게 완료되었을 때
        final_summary = "".join(summary_chunks).strip()
        llm_summaries.append(final_summary)
        print("\n" + "-"*50)
        
    except Exception as e:
        print(f"\n❌ 시스템 에러 발생: {e}")
        llm_summaries.append("시스템 예외 에러")
        print("-" * 50)

test_df['llm_summary'] = llm_summaries
print("\n🎉 모든 앵무새 무한루트 원천 차단형 스트리밍 요약 리포트 완료!")

test_df


총 30개의 요약 작업을 [파이썬 스트리밍 + 앵무새 무한루트 완벽 차단] 모드로 시작합니다!

▶ [1/30] 요약 중: 월드 랩스, '세계 생성' 능력 강화한 월드 모델 '마블 1.1' 출시
🤖 상태: 페이페이 리의 월드 랩스가 3D 세계 생성 모델 ‘마블 1.1’과 ‘마블 1.1 플러스’를 출시하며 3D 생성 기술의 발전을 이끌고 있다. 특히 ‘마블 1.1 플러스’는 기존 모델과 달리 장면에 맞춰 3D 공간을 자동으로 확장하여 더욱 크고 복잡한 세계를 생성하는 능력을 갖췄다. 이러한 기술 발전은 게임, 메타버스, 시뮬레이션 분야에서 맵 디자인 방식을 혁신하고, 사용자가 원하는 세계를 더욱 쉽게 구축할 수 있도록 도울 것으로 기대된다.
--------------------------------------------------

▶ [2/30] 요약 중: 넷플릭스, 물리 기반 AI 영상 모델 '보이드' 오픈소스 공개
🤖 상태: 넷플릭스는 영상 내 객체 제거 시 물리적 자연스러움까지 구현하는 AI 모델 '보이드'를 오픈소스로 공개했다. 보이드 프레임워크는 비전-언어 모델과 영상 확산 모델을 결합하여 객체 제거와 함께 그로 인한 장면의 물리적 변화까지 자동으로 조정한다. 넷플릭스는 시뮬레이션 기반 데이터셋을 활용하여 AI가 물리적 인과 관계를 학습하도록 함으로써, 기존 기술보다 시간적·물리적 일관성을 높였다.
--------------------------------------------------

▶ [3/30] 요약 중: 박정국 엘리스그룹 CTO '데이터센터, 규모보다 운영 역량이 더 중요'
🤖 상태: 엘리스그룹은 AI 데이터센터의 핵심 경쟁력이 규모가 아닌 빠른 구축과 GPU 자원 활용률을 높이는 운영 역량에 있다고 강조했다. 기존 데이터센터의 한계를 극복하기 위해 모듈형 데이터센터(PMDC)를 상용화했으며, 특히 아시아 최초로 수랭식 냉각 방식을 도입하여 효율성을 높였다. 엘리스그룹은 설계 단계부터 GPU 효율화에 집중한 엔드투엔드 서비스를 제공하며, 

,title,context,hybrid_summary,llm_summary
0,"월드 랩스, '세계 생성' 능력 강화한 월드 모델 '마블 1.1' 출시","페이페이 리의 월드 랩스가 새로운 3D 세계 생성 모델을 공개하며, 3D 생성 기술...","페이페이 리의 월드 랩스가 새로운 3D 세계 생성 모델을 공개하며, 3D 생성 기술...",페이페이 리의 월드 랩스가 3D 세계 생성 모델 ‘마블 1.1’과 ‘마블 1.1 플...
1,"넷플릭스, 물리 기반 AI 영상 모델 '보이드' 오픈소스 공개","넷플릭스가 영상 속 특정 객체를 제거할 때 단순한 배경 복원이 아니라, 물리적으로 ...","넷플릭스가 영상 속 특정 객체를 제거할 때 단순한 배경 복원이 아니라, 물리적으로 ...",넷플릭스는 영상 내 객체 제거 시 물리적 자연스러움까지 구현하는 AI 모델 '보이드...
2,"박정국 엘리스그룹 CTO '데이터센터, 규모보다 운영 역량이 더 중요'","“AI 데이터센터의 성패는 '규모'가 아니라, 얼마나 빨리 짓고 GPU 자원을 10...",하지만 기존 빌딩형 데이터센터는 조 단위 천문학적인 비용과 3년 이상의 건축 기간으...,엘리스그룹은 AI 데이터센터의 핵심 경쟁력이 규모가 아닌 빠른 구축과 GPU 자원 ...
3,"고재준 KTR 사업단장 “국내 최초 AI 인증, 기업 경쟁력에 날개 달아줄 것”",AI 도입이 가속화되는 가운데 신뢰성 확보가 기업의 핵심 과제로 부상하고 있다. 잘...,KTR(한국화학융합시험연구원)은 AI의 특성을 고려해 국내 최초로 국제 표준 기반 ...,AI 도입 확산과 함께 기업들은 AI 신뢰성 확보를 위한 객관적인 검증 체계 구축에...
4,이란 공격으로 바레인·두바이의 AWS 데이터센터 마비,이란이 중동 지역의 주요 데이터센터를 타격하면서 글로벌 클라우드 서비스와 반도체 공...,이란이 중동 지역의 주요 데이터센터를 타격하면서 글로벌 클라우드 서비스와 반도체 공...,이란의 데이터센터 공격으로 중동 지역의 클라우드 서비스와 반도체 공급망에 차질이 발...
5,데이터 전문 머코어 해킹으로 주요 AI 기업 '학습 비밀' 유출 비상,세계 최고 수준의 AI 모델 학습 기밀이 담긴 설계도가 유출됐을 가능성이 제기됐다....,이번 사태 이후 유명 해커그룹 랩서스(Lapsus$)는 머코어 유출본이라고 주장하는...,"AI 모델 학습 기밀 설계도가 유출될 가능성이 제기되며, 메타와 오픈AI의 협력사인..."
6,클로드 월 구독제로 '오픈클로' 못 쓴다...'시스템에 과도한 부담',앤트로픽이 수요 급증에 대응하기 위해 인기 AI 에이전트 '오픈클로'에 대한 '클로...,앤트로픽이 수요 급증에 대응하기 위해 인기 AI 에이전트 '오픈클로'에 대한 '클로...,앤트로픽은 급증하는 수요에 대응하기 위해 AI 에이전트 오픈클로의 클로드 구독 지원...
7,"아르시, 추론 모델 ‘트리니티-라지-싱킹’ 출시...에이전트 성능은 '클로드급'",미국의 대표 오픈소스 AI 스타트업인 아르시 AI(Arcee AI)가 다단계 추론이...,미국의 대표 오픈소스 AI 스타트업인 아르시 AI(Arcee AI)가 다단계 추론이...,미국의 아르시 AI가 다단계 추론이 가능한 첨단 에이전트 모델 ‘트리니티-라지-싱킹...
8,"알리바바, 차세대 영상 모델 ‘완2.7-비디오’ 공개","알리바바가 영상 생성부터 편집, 재구성까지 실제 영상 제작 전 과정을 통합한 차세대...","알리바바가 영상 생성부터 편집, 재구성까지 실제 영상 제작 전 과정을 통합한 차세대...","알리바바는 텍스트, 이미지, 영상, 음성 등 다양한 입력을 처리하는 멀티모달 AI ..."
9,"MS, 일본 AI 시장 확대 위해 15조 투자...AI 인재 100만명 양성",마이크로소프트(MS)가 일본 AI 시장 공략을 위해 대규모 투자에 나선다. MS는 ...,마이크로소프트(MS)가 일본 AI 시장 공략을 위해 대규모 투자에 나선다. MS는 ...,마이크로소프트는 일본 AI 시장 공략을 위해 2026년부터 2029년까지 약 15조...
